# APK vs Source-Code Library Comparison

This notebook compares libraries extracted from APK analysis against libraries extracted from source code.

Source-code analysis is treated as the ground truth. The primary match key is `pkg_name`; within each package, libraries are matched by normalized `library_key` and `library_name`.


In [14]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

SOURCE_CODE_PATH = Path('../database/source_code_libraries.csv')
APK_DATASET_PATH = Path('../analysis_datasets/dataset.csv')

source_df = pd.read_csv(SOURCE_CODE_PATH, dtype=str).fillna('')
apk_df = pd.read_csv(APK_DATASET_PATH, dtype=str).fillna('')

print(f'Source-code rows: {len(source_df):,}')
print(f'APK-analysis rows: {len(apk_df):,}')
print(f'Source packages: {source_df["pkg_name"].nunique():,}')
print(f'APK packages: {apk_df["pkg_name"].nunique():,}')


The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.
Source-code rows: 6,479
APK-analysis rows: 15,353
Source packages: 100
APK packages: 100


In [21]:
apk_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 15353 entries, 0 to 15352
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   pkg_name           15353 non-null  str  
 1   repo_host          15353 non-null  str  
 2   repo_path          15353 non-null  str  
 3   repo_url           15353 non-null  str  
 4   cited              15353 non-null  str  
 5   origin             15353 non-null  str  
 6   libarary_key       15353 non-null  str  
 7   library_name       15353 non-null  str  
 8   smali_prefix       15353 non-null  str  
 9   fingerprint_types  15353 non-null  str  
 10  classes_matched    15353 non-null  str  
 11  sample_class       15353 non-null  str  
 12  sample_class_file  15353 non-null  str  
dtypes: str(13)
memory usage: 7.0 MB


In [26]:
apk_df.groupby("pkg_name")["repo_url"].nunique().reset_index(name="distinct_repo_url_count").describe()

,distinct_repo_url_count
count,100.000000
mean,13.630000
std,7.816914
min,1.000000
25%,8.000000
50%,11.500000
75%,18.000000
max,44.000000


## Normalize identifiers

The APK dataset currently uses `libarary_key` while the source-code dataset uses `library_key`. This notebook normalizes both into shared comparison columns.


In [2]:
def normalize_text(value):
    value = str(value or '').strip().lower()
    value = value.replace('https://github.com/', '')
    value = value.replace('http://github.com/', '')
    value = value.removesuffix('.git')
    value = re.sub(r'[^a-z0-9._:/-]+', '', value)
    return value.strip('/ ')


def normalize_library_name(value):
    value = normalize_text(value)
    value = value.removesuffix('-android')
    value = value.removesuffix('.android')
    return value


def first_non_empty(*values):
    for value in values:
        value = str(value or '').strip()
        if value:
            return value
    return ''


def prepare_source(df):
    df = df.copy()
    for col in ['pkg_name', 'library_key', 'library_name', 'dependency_group', 'dependency_artifact', 'repo_path', 'repo_url']:
        if col not in df.columns:
            df[col] = ''

    df['norm_pkg'] = df['pkg_name'].map(normalize_text)
    df['norm_library_key'] = df['library_key'].map(normalize_text)
    df['norm_library_name'] = df['library_name'].map(normalize_library_name)
    df['norm_repo_path'] = df['repo_path'].map(normalize_text)
    df['norm_repo_url'] = df['repo_url'].map(normalize_text)
    df['norm_gradle_key'] = (
        df['dependency_group'].map(normalize_text) + ':' + df['dependency_artifact'].map(normalize_text)
    ).str.strip(':')

    df['match_key'] = df.apply(
        lambda r: first_non_empty(
            r['norm_repo_path'],
            r['norm_repo_url'],
            r['norm_library_key'],
            r['norm_gradle_key'],
            r['norm_library_name'],
        ),
        axis=1,
    )
    return df[df['norm_pkg'].ne('') & df['match_key'].ne('')]


def prepare_apk(df):
    df = df.copy()
    if 'library_key' not in df.columns and 'libarary_key' in df.columns:
        df['library_key'] = df['libarary_key']

    for col in ['pkg_name', 'library_key', 'library_name', 'repo_path', 'repo_url']:
        if col not in df.columns:
            df[col] = ''

    df['norm_pkg'] = df['pkg_name'].map(normalize_text)
    df['norm_library_key'] = df['library_key'].map(normalize_text)
    df['norm_library_name'] = df['library_name'].map(normalize_library_name)
    df['norm_repo_path'] = df['repo_path'].map(normalize_text)
    df['norm_repo_url'] = df['repo_url'].map(normalize_text)

    df['match_key'] = df.apply(
        lambda r: first_non_empty(
            r['norm_repo_path'],
            r['norm_repo_url'],
            r['norm_library_key'],
            r['norm_library_name'],
        ),
        axis=1,
    )
    return df[df['norm_pkg'].ne('') & df['match_key'].ne('')]

source_clean = prepare_source(source_df)
apk_clean = prepare_apk(apk_df)

print(f'Comparable source-code rows: {len(source_clean):,}')
print(f'Comparable APK rows: {len(apk_clean):,}')


Comparable source-code rows: 6,479
Comparable APK rows: 15,353


## Contains-based matching

The first-level identifier is `pkg_name`. Within the same package, a source-code library is counted as matched when any normalized source-code `library_key` or `library_name` is contained in any APK `library_key` or `library_name`, or vice versa. Repo/path fields are also included when available.

In [8]:
import re
from urllib.parse import urlparse


STOPWORDS = {
    '',
    'lib',
    'library',
    'android',
    'java',
    'kotlin',
    'github',
    'com',
    'org',
    'net',
    'io',
    'app',
    'core',
}


def clean_value(value):
    value = str(value or '').strip().lower()

    if value in {'nan', 'none', 'null'}:
        return ''

    value = value.replace('http://', 'https://')
    value = value.replace('https://www.github.com/', 'https://github.com/')
    value = value.replace('http://www.github.com/', 'https://github.com/')
    value = value.replace('www.github.com/', 'github.com/')

    value = value.replace('.git', '')
    value = value.strip().strip('/').strip()

    return value


def is_meaningful_key(value):
    value = clean_value(value)
    return len(value) >= 3 and value not in STOPWORDS


def normalize_repo_value(value):
    value = clean_value(value)

    if not value:
        return ''

    value = value.replace('https://github.com/', '')
    value = value.replace('http://github.com/', '')
    value = value.replace('github.com/', '')

    parts = [p for p in value.split('/') if p]

    if len(parts) >= 2:
        return f'{parts[0]}/{parts[1]}'

    return value


def normalize_text_key(value):
    value = clean_value(value)

    value = value.replace('https://github.com/', '')
    value = value.replace('github.com/', '')

    value = re.sub(r'[^a-z0-9]+', '.', value)
    value = re.sub(r'\.+', '.', value)
    value = value.strip('.')

    return value


def split_tokens(value):
    value = normalize_text_key(value)

    raw_tokens = re.split(r'[.\-_:\/]+', value)
    tokens = []

    for token in raw_tokens:
        token = token.strip()
        if len(token) >= 3 and token not in STOPWORDS:
            tokens.append(token)

    return set(tokens)


def artifact_part(value):
    value = clean_value(value)

    if ':' in value:
        parts = [p for p in value.split(':') if p]
        if len(parts) >= 2:
            return parts[1]

    if '/' in value:
        parts = [p for p in value.split('/') if p]
        if len(parts) >= 2:
            return parts[-1]

    value = normalize_text_key(value)
    parts = [p for p in re.split(r'[.\-_:\/]+', value) if p]

    if parts:
        return parts[-1]

    return value


def contains_match(source_value, apk_value):
    source_value = normalize_text_key(source_value)
    apk_value = normalize_text_key(apk_value)

    if not is_meaningful_key(source_value) or not is_meaningful_key(apk_value):
        return False

    return source_value in apk_value or apk_value in source_value


def repo_url_match(source_value, apk_value):
    source_repo = normalize_repo_value(source_value)
    apk_repo = normalize_repo_value(apk_value)

    if not is_meaningful_key(source_repo) or not is_meaningful_key(apk_repo):
        return False

    if source_repo == apk_repo:
        return True

    return source_repo in apk_repo or apk_repo in source_repo


def artifact_match(source_value, apk_value):
    source_artifact = artifact_part(source_value)
    apk_artifact = artifact_part(apk_value)

    if not is_meaningful_key(source_artifact) or not is_meaningful_key(apk_artifact):
        return False

    if source_artifact == apk_artifact:
        return True

    return source_artifact in apk_artifact or apk_artifact in source_artifact


def token_overlap_match(source_value, apk_value, min_overlap=1):
    source_tokens = split_tokens(source_value)
    apk_tokens = split_tokens(apk_value)

    if not source_tokens or not apk_tokens:
        return False

    overlap = source_tokens.intersection(apk_tokens)

    if len(overlap) >= min_overlap:
        return True

    return False


def row_matches(source_row, apk_row):
    source_repo_candidates = [
        source_row.get('norm_repo_url', ''),
        source_row.get('repo_url', ''),
        source_row.get('thirdparty_url', ''),
        source_row.get('norm_repo_path', ''),
        source_row.get('repo_path', ''),
    ]

    apk_repo_candidates = [
        apk_row.get('norm_repo_url', ''),
        apk_row.get('repo_url', ''),
        apk_row.get('norm_repo_path', ''),
        apk_row.get('repo_path', ''),
    ]

    source_candidates = [
        source_row.get('norm_library_key', ''),
        source_row.get('library_key', ''),
        source_row.get('norm_library_name', ''),
        source_row.get('library_name', ''),
        source_row.get('norm_gradle_key', ''),
        source_row.get('dependency_group', ''),
        source_row.get('dependency_artifact', ''),
        source_row.get('norm_repo_path', ''),
        source_row.get('norm_repo_url', ''),
        source_row.get('repo_path', ''),
        source_row.get('repo_url', ''),
        source_row.get('thirdparty_url', ''),
        source_row.get('match_key', ''),
        source_row.get('declaration', ''),
    ]

    apk_candidates = [
        apk_row.get('norm_library_key', ''),
        apk_row.get('library_key', ''),
        apk_row.get('libarary_key', ''),
        apk_row.get('norm_library_name', ''),
        apk_row.get('library_name', ''),
        apk_row.get('norm_repo_path', ''),
        apk_row.get('norm_repo_url', ''),
        apk_row.get('repo_path', ''),
        apk_row.get('repo_url', ''),
        apk_row.get('match_key', ''),
    ]

    for source_value in source_repo_candidates:
        for apk_value in apk_repo_candidates:
            if repo_url_match(source_value, apk_value):
                return True, source_value, apk_value, 'repo_url_match'

    for source_value in source_candidates:
        for apk_value in apk_candidates:
            if contains_match(source_value, apk_value):
                return True, source_value, apk_value, 'contains_match'

    for source_value in source_candidates:
        for apk_value in apk_candidates:
            if artifact_match(source_value, apk_value):
                return True, source_value, apk_value, 'artifact_match'

    for source_value in source_candidates:
        for apk_value in apk_candidates:
            if token_overlap_match(source_value, apk_value, min_overlap=1):
                return True, source_value, apk_value, 'token_overlap_match'

    return False, '', '', ''


source_records = source_clean.reset_index(drop=True).reset_index().rename(columns={'index': 'source_id'})
apk_records = apk_clean.reset_index(drop=True).reset_index().rename(columns={'index': 'apk_id'})

match_rows = []

for pkg, source_group in source_records.groupby('norm_pkg'):
    apk_group = apk_records[apk_records['norm_pkg'] == pkg]
    counter = 0

    if apk_group.empty:
        continue

    for _, source_row in source_group.iterrows():
        for _, apk_row in apk_group.iterrows():
            matched, source_match_value, apk_match_value, match_type = row_matches(source_row, apk_row)

            if matched:
                match_rows.append({
                    'norm_pkg': pkg,
                    'source_id': source_row['source_id'],
                    'apk_id': apk_row['apk_id'],
                    'source_match_value': source_match_value,
                    'apk_match_value': apk_match_value,
                    'match_type': match_type,
                })
                counter += 1

    print(f"{counter} matched rows for {pkg}")

matches = pd.DataFrame(match_rows)

matched_source_ids = set(matches['source_id']) if not matches.empty else set()
matched_apk_ids = set(matches['apk_id']) if not matches.empty else set()

source_eval = source_records.copy()
source_eval['in_source'] = True
source_eval['in_apk'] = source_eval['source_id'].isin(matched_source_ids)
source_eval['status'] = np.where(source_eval['in_apk'], 'matched', 'missed_by_apk')
source_eval['comparison_id'] = 'source_' + source_eval['source_id'].astype(str)

apk_extra_eval = apk_records[~apk_records['apk_id'].isin(matched_apk_ids)].copy()
apk_extra_eval['in_source'] = False
apk_extra_eval['in_apk'] = True
apk_extra_eval['status'] = 'extra_in_apk'
apk_extra_eval['comparison_id'] = 'apk_' + apk_extra_eval['apk_id'].astype(str)

comparison = pd.concat([
    source_eval[['comparison_id', 'norm_pkg', 'match_key', 'in_source', 'in_apk', 'status', 'source_id']],
    apk_extra_eval[['comparison_id', 'norm_pkg', 'match_key', 'in_source', 'in_apk', 'status', 'apk_id']],
], ignore_index=True, sort=False)

comparison['pkg_name'] = comparison['norm_pkg']

print(f'Matched source rows: {len(matched_source_ids):,}')
print(f'Matched APK rows: {len(matched_apk_ids):,}')
print(f'Match relation rows: {len(matches):,}')

if not matches.empty:
    print(matches['match_type'].value_counts())

comparison.head()

0 matched rows for a2dp.vol
1108 matched rows for app.michaelwuensch.bitbanana
875 matched rows for app.organicmaps
742 matched rows for app.zeusln.zeus
2 matched rows for at.jclehner.rxdroid
595 matched rows for au.id.micolous.farebot
22 matched rows for br.com.colman.petals
41 matched rows for btools.routingapp
4103 matched rows for chat.schildi.android
699 matched rows for chat.simplex.app
1587 matched rows for com.afkanerd.deku
1814 matched rows for com.afkanerd.sw0b
94 matched rows for com.aicodix.rattlegram
321 matched rows for com.akylas.cardwallet
179 matched rows for com.arduia.expense
0 matched rows for com.blockbasti.justanotherworkouttimer
6 matched rows for com.brentpanther.bitcoinwidget
9 matched rows for com.clearevo.bluetooth_gnss
130 matched rows for com.cointrend
0 matched rows for com.danilkinkin.buckwheat
1729 matched rows for com.databag
0 matched rows for com.derdilla.bloodpressureapp
4 matched rows for com.dreautall.waterflyiii
4 matched rows for com.ds.avare
192

,comparison_id,norm_pkg,match_key,in_source,in_apk,status,source_id,apk_id,pkg_name
0,source_0,com.github.shadowsocks,android-gradle,True,False,missed_by_apk,0.0,NaN,com.github.shadowsocks
1,source_1,com.github.shadowsocks,androidx-browser,True,True,matched,1.0,NaN,com.github.shadowsocks
2,source_2,com.github.shadowsocks,androidx-camera-camera2,True,True,matched,2.0,NaN,com.github.shadowsocks
3,source_3,com.github.shadowsocks,androidx-camera-lifecycle,True,True,matched,3.0,NaN,com.github.shadowsocks
4,source_4,com.github.shadowsocks,androidx-camera-view,True,True,matched,4.0,NaN,com.github.shadowsocks


In [11]:
matches[matches['match_type'] == 'token_overlap_match'].tail(50)

,norm_pkg,source_id,apk_id,source_match_value,apk_match_value,match_type
106147,xyz.zood.george,6013,15266,androidx.work:work-runtime,androidx/constraintlayout,token_overlap_match
106148,xyz.zood.george,6013,15267,androidx.work:work-runtime,androidx/constraintlayout,token_overlap_match
106149,xyz.zood.george,6013,15268,androidx.work:work-runtime,androidx/constraintlayout,token_overlap_match
106150,xyz.zood.george,6013,15269,androidx.work:work-runtime,androidx/constraintlayout,token_overlap_match
106151,xyz.zood.george,6013,15270,androidx.work:work-runtime,androidx/constraintlayout,token_overlap_match
106152,xyz.zood.george,6013,15271,androidx.work:work-runtime,androidx/constraintlayout,token_overlap_match
106153,xyz.zood.george,6013,15272,androidx.work:work-runtime,androidx/constraintlayout,token_overlap_match
106154,xyz.zood.george,6013,15273,androidx.work:work-runtime,androidx/constraintlayout,token_overlap_match
106155,xyz.zood.george,6013,15274,androidx.work:work-runtime,androidx/constraintlayout,token_overlap_match
106156,xyz.zood.george,6013,15275,androidx.work:work-runtime,androidx/constraintlayout,token_overlap_match


## Overall accuracy metrics

Because source-code analysis is ground truth:

- **True Positive (TP):** library exists in source code and APK analysis found it.
- **False Negative (FN):** library exists in source code but APK analysis missed it.
- **False Positive (FP):** APK analysis found a library not found in source code.
- **Recall / Ground-truth coverage:** how much of the source-code ground truth APK analysis detected.
- **Precision:** how much of APK analysis is supported by source-code ground truth.
- **F1:** balanced precision/recall score.


In [9]:
tp = int(((comparison['in_source']) & (comparison['in_apk'])).sum())
fn = int(((comparison['in_source']) & (~comparison['in_apk'])).sum())
fp = int(((~comparison['in_source']) & (comparison['in_apk'])).sum())

recall = tp / (tp + fn) if (tp + fn) else 0
precision = tp / (tp + fp) if (tp + fp) else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0
jaccard = tp / (tp + fp + fn) if (tp + fp + fn) else 0

metrics = pd.DataFrame([
    {'metric': 'true_positive_matched', 'value': tp},
    {'metric': 'false_negative_missed_by_apk', 'value': fn},
    {'metric': 'false_positive_extra_in_apk', 'value': fp},
    {'metric': 'recall_ground_truth_coverage', 'value': recall},
    {'metric': 'precision_apk_supported_by_source', 'value': precision},
    {'metric': 'f1_score', 'value': f1},
    {'metric': 'jaccard_overlap', 'value': jaccard},
])

metrics


,metric,value
0,true_positive_matched,2741.000000
1,false_negative_missed_by_apk,3738.000000
2,false_positive_extra_in_apk,3613.000000
3,recall_ground_truth_coverage,0.423059
4,precision_apk_supported_by_source,0.431382
5,f1_score,0.427180
6,jaccard_overlap,0.271601


## Per-package comparison


In [13]:
per_package = comparison.groupby('pkg_name').agg(
    source_libraries=('in_source', 'sum'),
    apk_libraries=('in_apk', 'sum'),
    matched=('status', lambda s: (s == 'matched').sum()),
    missed_by_apk=('status', lambda s: (s == 'missed_by_apk').sum()),
    extra_in_apk=('status', lambda s: (s == 'extra_in_apk').sum()),
).reset_index()

per_package['recall'] = per_package['matched'] / per_package['source_libraries'].replace(0, np.nan)
per_package['precision'] = per_package['matched'] / per_package['apk_libraries'].replace(0, np.nan)
per_package['f1'] = 2 * per_package['precision'] * per_package['recall'] / (per_package['precision'] + per_package['recall'])
per_package = per_package.fillna(0).sort_values(['recall', 'matched'], ascending=[True, False])

per_package.head(50)


,pkg_name,source_libraries,apk_libraries,matched,missed_by_apk,extra_in_apk,recall,precision,f1
0,a2dp.vol,5,4,0,5,4,0.000000,0.000000,0.000000
15,com.blockbasti.justanotherworkouttimer,1,9,0,1,9,0.000000,0.000000,0.000000
19,com.danilkinkin.buckwheat,43,8,0,43,8,0.000000,0.000000,0.000000
21,com.derdilla.bloodpressureapp,9,9,0,9,9,0.000000,0.000000,0.000000
40,com.powerpoint45.lucidbrowser,5,127,0,5,127,0.000000,0.000000,0.000000
43,com.ripster.sossoldi,6,7,0,6,7,0.000000,0.000000,0.000000
50,com.zulipmobile,37,10,0,37,10,0.000000,0.000000,0.000000
58,de.wger.flutter,18,14,0,18,14,0.000000,0.000000,0.000000
65,info.zverev.ilya.every_door,20,28,0,20,28,0.000000,0.000000,0.000000
77,om.sstvencoder,3,27,0,3,27,0.000000,0.000000,0.000000


## Detailed matched, missed, and extra libraries


In [14]:

source_detail_cols = ['source_id', 'norm_pkg', 'match_key', 'pkg_name', 'library_key', 'library_name', 'method', 'file_path', 'declaration', 'repo_url', 'thirdparty_url']
apk_detail_cols = ['apk_id', 'norm_pkg', 'match_key', 'pkg_name', 'library_key', 'library_name', 'repo_url', 'repo_path', 'origin', 'fingerprint_types', 'classes_matched', 'sample_class']

source_detail_cols = [c for c in source_detail_cols if c in source_records.columns]
apk_detail_cols = [c for c in apk_detail_cols if c in apk_records.columns]

if not matches.empty:
    matched_details = matches.merge(
        source_records[source_detail_cols],
        on=['source_id', 'norm_pkg'],
        how='left'
    ).merge(
        apk_records[apk_detail_cols],
        on=['apk_id', 'norm_pkg'],
        how='left',
        suffixes=('_source', '_apk')
    )
else:
    matched_details = pd.DataFrame(columns=['norm_pkg', 'source_id', 'apk_id', 'source_match_value', 'apk_match_value'])

missed_by_apk_details = source_records[~source_records['source_id'].isin(matched_source_ids)][source_detail_cols].copy()
extra_in_apk_details = apk_records[~apk_records['apk_id'].isin(matched_apk_ids)][apk_detail_cols].copy()

print(f'Matched detail rows: {len(matched_details):,}')
print(f'Missed by APK detail rows: {len(missed_by_apk_details):,}')
print(f'Extra in APK detail rows: {len(extra_in_apk_details):,}')


Matched detail rows: 23,029
Missed by APK detail rows: 5,754
Extra in APK detail rows: 6,519


In [8]:
matched_details

,norm_pkg,source_id,apk_id,source_match_value,apk_match_value,match_key_source,pkg_name_source,library_key_source,library_name_source,method,...,match_key_apk,pkg_name_apk,library_key_apk,library_name_apk,repo_url_apk,repo_path,origin,fingerprint_types,classes_matched,sample_class
0,app.michaelwuensch.bitbanana,3673,32,android,androidx/constraintlayout,io.github.zxing-cpp:android,app.michaelwuensch.bitbanana,io.github.zxing-cpp:android,android,gradle_dependency,...,androidx/constraintlayout,app.michaelwuensch.bitbanana,androidx/constraintlayout,constraintlayout,https://github.com/androidx/constraintlayout,androidx/constraintlayout,fingerprints,java_package,258,Landroidx/constraintlayout/core/ArrayLinkedVar...
1,app.michaelwuensch.bitbanana,3673,33,android,androidx/constraintlayout,io.github.zxing-cpp:android,app.michaelwuensch.bitbanana,io.github.zxing-cpp:android,android,gradle_dependency,...,androidx/constraintlayout,app.michaelwuensch.bitbanana,androidx/constraintlayout,constraintlayout,https://github.com/androidx/constraintlayout,androidx/constraintlayout,fingerprints,java_package,77,Landroidx/constraintlayout/core/motion/CustomA...
2,app.michaelwuensch.bitbanana,3673,34,android,androidx/constraintlayout,io.github.zxing-cpp:android,app.michaelwuensch.bitbanana,io.github.zxing-cpp:android,android,gradle_dependency,...,androidx/constraintlayout,app.michaelwuensch.bitbanana,androidx/constraintlayout,constraintlayout,https://github.com/androidx/constraintlayout,androidx/constraintlayout,fingerprints,java_package,13,Landroidx/constraintlayout/core/parser/CLArray;
3,app.michaelwuensch.bitbanana,3673,35,android,androidx/constraintlayout,io.github.zxing-cpp:android,app.michaelwuensch.bitbanana,io.github.zxing-cpp:android,android,gradle_dependency,...,androidx/constraintlayout,app.michaelwuensch.bitbanana,androidx/constraintlayout,constraintlayout,https://github.com/androidx/constraintlayout,androidx/constraintlayout,fingerprints,java_package,49,Landroidx/constraintlayout/core/dsl/Barrier;
4,app.michaelwuensch.bitbanana,3673,36,android,androidx/constraintlayout,io.github.zxing-cpp:android,app.michaelwuensch.bitbanana,io.github.zxing-cpp:android,android,gradle_dependency,...,androidx/constraintlayout,app.michaelwuensch.bitbanana,androidx/constraintlayout,constraintlayout,https://github.com/androidx/constraintlayout,androidx/constraintlayout,fingerprints,java_package,44,Landroidx/constraintlayout/core/widgets/Barrier;
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9827,xyz.zood.george,3576,15349,com.squareup.retrofit2:converter-gson,retrofit,com.squareup.retrofit2:converter-gson,xyz.zood.george,com.squareup.retrofit2:converter-gson,converter-gson,gradle_dependency,...,square/retrofit,xyz.zood.george,square/retrofit,retrofit,https://github.com/square/retrofit,square/retrofit,fingerprints,java_package,1,Lretrofit2/internal/EverythingIsNonNull;
9828,xyz.zood.george,3576,15350,com.squareup.retrofit2:converter-gson,retrofit,com.squareup.retrofit2:converter-gson,xyz.zood.george,com.squareup.retrofit2:converter-gson,converter-gson,gradle_dependency,...,square/retrofit,xyz.zood.george,square/retrofit,retrofit,https://github.com/square/retrofit,square/retrofit,fingerprints,java_package,25,Lretrofit2/http/Body;
9829,xyz.zood.george,3576,15351,com.squareup.retrofit2:converter-gson,retrofit,com.squareup.retrofit2:converter-gson,xyz.zood.george,com.squareup.retrofit2:converter-gson,converter-gson,gradle_dependency,...,square/retrofit,xyz.zood.george,square/retrofit,retrofit,https://github.com/square/retrofit,square/retrofit,fingerprints,java_package,5,Lretrofit2/converter/gson/GsonConverterFactory;
9830,xyz.zood.george,3578,15228,com.vanniktech:android-image-cropper,android-image-cropper,com.vanniktech:android-image-cropper,xyz.zood.george,com.vanniktech:android-image-cropper,android-image-cropper,gradle_dependency,...,canhub/android-image-cropper,xyz.zood.george,CanHub/Android-Image-Cropper,Android-Image-

In [ ]:
missed_by_apk_details.head(20)

In [ ]:
extra_in_apk_details.head(20)

## Export comparison outputs

The notebook writes CSV files under `../analysis_outputs/library_comparison/`.


In [ ]:
OUTPUT_DIR = Path('../analysis_outputs/library_comparison')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

metrics.to_csv(OUTPUT_DIR / 'overall_metrics.csv', index=False)
per_package.to_csv(OUTPUT_DIR / 'per_package_metrics.csv', index=False)
comparison.to_csv(OUTPUT_DIR / 'package_library_comparison.csv', index=False)
matched_details.to_csv(OUTPUT_DIR / 'matched_details.csv', index=False)
missed_by_apk_details.to_csv(OUTPUT_DIR / 'missed_by_apk_details.csv', index=False)
extra_in_apk_details.to_csv(OUTPUT_DIR / 'extra_in_apk_details.csv', index=False)

print(f'Outputs written to: {OUTPUT_DIR}')


## Optional: quick charts


In [ ]:
import matplotlib.pyplot as plt

status_counts = comparison['status'].value_counts().reindex(['matched', 'missed_by_apk', 'extra_in_apk']).fillna(0)
ax = status_counts.plot(kind='bar')
ax.set_title('APK vs Source-Code Library Comparison')
ax.set_xlabel('Status')
ax.set_ylabel('Unique package-library pairs')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()
